# Google Trends Contrarian on BTC

Contrarian attention model. Short BTC when 'bitcoin' search interest is unusually high (4-week z-score > 1), long when unusually low (z < -1), flat otherwise. Weekly resolution, 2021-2024.

See [`README.md`](README.md) for the writeup and [`backtest.py`](backtest.py) for the reproducible CLI version.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from tradinglib.backtest import run_backtest
from tradinglib.loaders.equities.yfinance import load_daily
from tradinglib.loaders.sentiment.google_trends import load_interest

QUERY = "bitcoin"
SYMBOL = "BTC-USD"
TIMEFRAME = "2021-01-01 2024-12-31"
START, END = "2021-01-01", "2024-12-31"

## Load and align weekly data

In [ ]:
daily = load_daily(SYMBOL, start=START, end=END)
btc_weekly = daily["close"].resample("W-SUN").last().dropna()
interest = load_interest(QUERY, timeframe=TIMEFRAME)

common = btc_weekly.index.intersection(interest.index)
btc_weekly = btc_weekly.loc[common]
interest = interest.loc[common]
print(f"{len(common)} aligned weekly bars from {common[0].date()} to {common[-1].date()}")

## Inspect: are price and search interest correlated?

In [ ]:
joined = pd.DataFrame({"btc": btc_weekly, "interest": interest})
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
joined["btc"].plot(ax=ax1, color="black", label="BTC weekly close")
joined["interest"].plot(ax=ax2, color="steelblue", alpha=0.6, label="'bitcoin' search interest")
ax1.set_ylabel("BTC price ($)")
ax2.set_ylabel("Search interest (0-100)")
ax1.set_title("BTC price and Google Trends 'bitcoin' search interest")
ax1.grid(True, alpha=0.3)
plt.show()

print("Spearman corr (level):", joined.corr(method="spearman").iloc[0, 1].round(3))

## Build the z-score signal

In [ ]:
window = 4
mean = interest.rolling(window).mean()
std = interest.rolling(window).std()
z = (interest - mean) / std

signal = pd.Series(0.0, index=interest.index)
signal[z < -1.0] = 1.0
signal[z > 1.0] = -1.0
signal.value_counts().rename({-1.0: "short", 0.0: "flat", 1.0: "long"})

## Run the backtest

In [ ]:
result = run_backtest(btc_weekly, signal, fee_bps=2.0, slippage_bps=3.0, periods_per_year=52)
pd.Series(result.metrics)

## Equity curve vs buy & hold

In [ ]:
buy_hold = (1.0 + btc_weekly.pct_change().fillna(0.0)).cumprod() * result.config["initial_capital"]

fig, ax = plt.subplots(figsize=(12, 5))
result.equity_curve.plot(ax=ax, label="Trends contrarian (BTC)")
buy_hold.plot(ax=ax, label="Buy & hold (BTC weekly)", alpha=0.6)
ax.set_title("BTC \u2014 Google-Trends contrarian vs buy & hold (weekly)")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()